# Phase 3: LoRA fine-tuning of Qwen2-VL-2B on native-Windows GUI grounding (v3)

Third real training run for the `computer-use` project's GUI grounding model (ADR-0003). Runs on Kaggle's free T4/P100 GPU quota, matching the project's $0-budget commitment.

**Honest scope, stated up front**: this trains on the **v3 dataset** -- 3163 examples across 200 screenshots, all 14/14 registry apps, collected 2026-07-19/20 after (1) a wrong-window bug meant 36% of v2's rows were the code editor mislabeled as other apps, fixed with window-identity verification and an occlusion filter, and (2) a third real PII incident -- Windows 11 Notepad restoring a real `.env` tab from its own on-disk session state on a cold launch, a mechanism the existing `single_instance` guard couldn't see because it only checked for an *already-running* process. Both are root-caused and fixed in code (not just re-collected around), and the affected rows were quarantined -- see `data/gui_grounding/README.md`'s v3 section for the full writeup. The held-out pool still covers all 4 held-out apps (Character Map, Device Manager, Audacity, Notepad++), so H3's generalization claim has real diversity behind it -- but this run is still an engineering validation (does the fixed pipeline train end-to-end, is the loss sane), not the final ablation study.

Every piece below (`prepare_dataset.py`, `chat_format.py`, `lora_config.py`, `dataset.py`, `train_lora.py`) was built and verified locally (157 passing tests, plus live checks against the real tokenizer/processor/model architecture on the `meta` device) before this notebook was written -- see `docs/journal.md` for that verification trail. The v1/v2 runs already proved this notebook touches real model weights/GPU correctly; this run's job is to confirm the same holds on the corrected, PII-clean, full-coverage dataset.

## 1. Confirm GPU is attached

In Kaggle: Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100). Must be set before running anything below.

In [ ]:
!nvidia-smi

In [ ]:
import os

# Must be set before torch/cuda initializes in this process -- restrict to
# a single GPU. HF Trainer auto-wraps the model in plain nn.DataParallel
# whenever more than one GPU is visible (Kaggle's "T4 x2" option gives you
# two); DataParallel empties .parameters() on each replica for its autograd
# trick, which breaks Qwen2-VL's self.visual.dtype (computed lazily via
# next(p.dtype for p in self.parameters() if ...)) with a StopIteration
# deep in the forward pass. Single GPU also matches the batch-size math
# this notebook already documents (per_device=1 x grad_accum=4 = effective
# batch 4 -- that's a one-GPU number, two GPUs would silently double it).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 2. Get the code

Clones the public `computer-use` repo and installs it with the `training` extra (`transformers`, `torch`, `torchvision`, `peft`, `jinja2` -- see `pyproject.toml`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")

# Idempotent: a plain `!git clone` errors out ("destination path already
# exists and is not empty") if this cell is re-run without a kernel restart
# -- e.g. after fixing something downstream. Skipping the clone when the dir
# is already there makes re-running this cell always safe, in either case
# (fresh session or same session, re-run).
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git
else:
    print(f"{REPO_DIR} already present in this session, skipping clone")

os.chdir(REPO_DIR)
# Use {sys.executable} -m pip, not bare `pip` -- Kaggle images can have more
# than one Python/pip on PATH, and a bare `!pip install` has been observed to
# install into a different environment than the one this kernel is actually
# running, producing "ModuleNotFoundError: No module named 'computeruse'" on
# the very next cell even though the install itself reported success.
# Also dropped `-q` here deliberately: a silent install failure (dependency
# conflict, etc.) is exactly what would cause the same downstream error, and
# `-q` was hiding whichever of the two causes was actually happening.
!{sys.executable} -m pip install -e ".[training]"

In [ ]:
import importlib
import sys
from pathlib import Path

# Belt-and-suspenders: the editable install's .pth-based finder is only
# picked up by a *fresh* interpreter (site.py reads .pth files once, at
# startup) -- if a previous cell in this same session already installed it,
# this is a no-op; if not, this makes the import work without a restart.
REPO_SRC = Path("/kaggle/working/computer-use/src")
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

importlib.invalidate_caches()
try:
    import computeruse
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"{e}. This means cell 2 (clone + pip install) never actually ran in "
        "this kernel session -- almost always because of a restart, or "
        "jumping to this cell out of order. Re-run the notebook from the top, "
        "in order, in the current session; do not skip ahead."
    ) from e
print("computeruse imported OK from", computeruse.__file__)

In [ ]:
import sys

# Kaggle's base image preinstalls torchao (0.10.0 as of 2026-07-16), and
# peft's LoRA dispatcher unconditionally checks its version -- if present
# but below peft's required minimum, it raises ImportError even though we
# never use torchao anywhere in this project. If it's simply not installed,
# peft's is_torchao_available() returns False and quietly falls through to
# the LoRA dispatch path we actually want, so uninstalling is the fix, not
# upgrading (confirmed working 2026-07-16).
!{sys.executable} -m pip uninstall -y -q torchao

## 3. Point at the dataset

Kaggle's actual mount path nests under `datasets/<owner>/<slug>`, not the flat `/kaggle/input/<slug>` the "+ Add Input" panel implies (confirmed 2026-07-16). Rather than hardcode either convention, the cell below *searches* for `labels.jsonl`.

It also asserts there is exactly **one**. This matters: if an older dataset version is still attached alongside the current one, training silently runs against whichever path is found first -- an entire wasted GPU run against stale data, with nothing in the output to indicate it. Detach every old version in the right-hand panel before running.


In [ ]:
from pathlib import Path

from computeruse.training.dataset import resolve_path

_candidates = sorted(Path("/kaggle/input").rglob("labels.jsonl"))
assert _candidates, (
    "no labels.jsonl found anywhere under /kaggle/input -- check the v3 "
    "dataset is actually attached via '+ Add Input' in the right panel"
)
assert len(_candidates) == 1, (
    f"found {len(_candidates)} labels.jsonl under /kaggle/input: {_candidates}. "
    "Detach the old/extra dataset versions in the right-hand panel. Training "
    "against the wrong one costs a full GPU run and looks identical in the logs."
)
KAGGLE_DATASET_ROOT = _candidates[0].parent
OUTPUT_DIR = Path("/kaggle/working/lora_grounder")

# resolve_path handles Kaggle's uploader flattening nested folders
train_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/train.jsonl")
dev_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/dev.jsonl")

n_train = len([ln for ln in train_split_path.read_text(encoding="utf-8").splitlines() if ln.strip()])
n_dev = len([ln for ln in dev_split_path.read_text(encoding="utf-8").splitlines() if ln.strip()])
print("dataset root:", KAGGLE_DATASET_ROOT)
print(f"train: {n_train} examples   dev: {n_dev} examples")
# v3 (full re-collection + PII fix, 2026-07-20) is 1572 train / 555 dev across
# 200 screenshots. Anything else means you are attached to a different
# dataset version than you think -- v2 was 252/146 over just 32 screenshots
# (36% of its rows were the wrong application entirely), and the pre-Notepad-
# fix interim state earlier the same day was 1425/479.
assert (n_train, n_dev) == (1572, 555), (
    f"expected the v3 dataset (1572 train / 555 dev), got {n_train}/{n_dev} -- "
    "this is almost certainly a stale dataset version still attached."
)

## 4. Build the trainer

Loads the real Qwen2-VL-2B-Instruct weights (~4GB download, first cell run only), attaches the verified LoRA config (0.829% trainable params against the real architecture -- see `training/lora_config.py`), and wires up `GroundingDataset`/`collate_fn` against the v3 train/dev splits.

In [ ]:
from pathlib import Path

from computeruse.training.train_lora import find_last_checkpoint

# Cross-run resume (added 2026-08-15, after losing four separate runs to
# session death/cancellation). Instead of one long risky run, this does
# short committed chunks -- each "Save & Run All (Commit)" only has to
# survive STEP_INCREMENT steps' worth of risk, so a mid-run failure loses
# at most one chunk, never everything banked so far.
#
# find_last_checkpoint(OUTPUT_DIR) alone only covers resuming *within* one
# still-running session -- OUTPUT_DIR is always empty at the start of a
# fresh commit, since each commit runs in a brand-new container. To
# continue in the next run: after this run finishes and commits, go to the
# right panel -> "+ Add Input" -> search your own username/this notebook's
# name, and attach its own previous output as an input. Kaggle mounts it
# read-only under /kaggle/input/<notebook-slug>/..., preserving the same
# lora_grounder/checkpoint-N structure OUTPUT_DIR had -- so this also
# searches there, same pattern as the labels.jsonl search above. A
# read-only mount is fine here: resume_from_checkpoint only ever reads
# from it, this run's own new checkpoints get written under OUTPUT_DIR.
_input_checkpoints = sorted(
    (p for p in Path("/kaggle/input").rglob("checkpoint-*") if p.is_dir()),
    key=lambda p: int(p.name.split("-")[1]),
)
resume_from = find_last_checkpoint(OUTPUT_DIR) or (
    str(_input_checkpoints[-1]) if _input_checkpoints else None
)
current_step = int(resume_from.rsplit("-", 1)[1]) if resume_from else 0

# Sized against Kaggle's 12h session ceiling, with margin, because the
# margin is the whole point. ~57s/step observed (2026-08-15), so 400 steps
# is ~6.3h of training + ~45min clone/install/model-download + ~20min of
# dev evals (555 examples every eval_steps=100) = roughly 7.5h. That leaves
# ~4.5h of headroom.
#
# Why not a bigger chunk: a commit that overruns the 12h ceiling is killed,
# and a killed commit's /kaggle/working is not reliably persisted -- which
# is precisely how runs were lost before. Finishing well inside the ceiling
# is worth more than finishing in fewer chunks.
#
# Three chunks (400 -> 800 -> 1179) complete all 3 epochs in ~22h of the
# 30h weekly quota. save_steps=100 underneath means even a chunk that dies
# unexpectedly loses at most ~100 steps, not the run.
STEP_INCREMENT = 400
MAX_STEPS = current_step + STEP_INCREMENT
# Print what the handoff actually found, not just the conclusion. A silently
# failed "+ Add Input" attach looks identical to a genuine first run --
# except it silently throws away every step banked so far.
print(f"checkpoints visible under /kaggle/input: {[p.name for p in _input_checkpoints]}")
if resume_from:
    print(f"resuming from step {current_step} (checkpoint: {resume_from})")
else:
    print("no checkpoint found -- starting fresh from step 0")
    print("  ^ if this is NOT your first chunk, STOP. The previous run's output")
    print("    is not attached: right panel -> + Add Input -> your username ->")
    print("    this notebook's previous version. Continuing now restarts at 0.")
print(f"this run targets step {MAX_STEPS} (of 1179 total for 3 epochs)")


In [ ]:
from computeruse.training.train_lora import build_trainer

trainer = build_trainer(KAGGLE_DATASET_ROOT, OUTPUT_DIR, max_steps=MAX_STEPS)

## 5. Train

**IMPORTANT -- read before running, this is what lost the last two runs' weights:** use Kaggle's **"Save & Run All (Commit)"**, not an interactively-clicked-through Draft session. A Draft session's filesystem (including every `save_steps=100` checkpoint under `OUTPUT_DIR`) lives only in that session's ephemeral container -- if the tab errors out, disconnects, or you just close the browser, everything under `/kaggle/working` is gone with it, even though the loss log you can still see in cell output is real. A committed run persists its `/kaggle/working` output regardless of what happens to the browser tab afterward. This is not optional for a run you intend to actually keep.

**Chunked runs, not one long attempt** (see the cell above): each run trains `STEP_INCREMENT=100` steps past wherever the last committed checkpoint left off, not the full 3-epoch/1179-step schedule in one shot. After this run finishes and commits, attach its own output as an Input (right panel -> "+ Add Input" -> your username/this notebook) before running again -- that's what lets the next run pick up from `current_step` instead of 0. Two things changed 2026-07-19 that still matter for reading the output:

- **Mixed precision is now real.** The previous runs loaded fp16 weights but never set `fp16=True`, so no `GradScaler` was attached and small gradients underflowed to zero unnoticed -- the adapter barely moved while the loss (dominated by the frozen base) still looked sane. Expect the loss curve to behave *differently* from previous runs; that is the fix working, not a regression.
- **The best-dev checkpoint is kept, not the last one** (`load_best_model_at_end`). On a few hundred examples, a later epoch can easily be worse than an earlier one.

Per the hypothesis doc, **only the dev split may drive tuning** -- do not look at `test_held_out_app` or `test_same_app` to make decisions here.


In [ ]:
train_result = trainer.train(resume_from_checkpoint=resume_from)
print(train_result)


In [ ]:
trainer.save_model(str(OUTPUT_DIR / "final"))
print("saved LoRA adapter to", OUTPUT_DIR / "final")

## 6. Evaluate on dev -- a real number, not a vibe check

The previous version of this cell generated on `dev_examples[:5]` and eyeballed the strings. That was actively misleading: `dev.jsonl` is grouped by app, so the first five are always the *same* five, and three of them are Calculator's Minimize/Restore/Close titlebar buttons -- adjacent targets 36 normalized units apart, visually near-identical, among the hardest in the whole split. Every run so far has been judged on the worst possible sample of five.

This runs the **full 555-example dev split** through `eval/vlm_grounder.py` instead and reports:

- **click accuracy** (predicted point lands inside the ground-truth box) -- the H1-H3 metric;
- **parse rate** -- did the model emit a readable `(x,y)` at all;
- **median center distance** in [0,1000) units -- the diagnostic that actually shows progress. The median target box is only ~40 units across, so accuracy is a nearly-all-zeros signal early on: a model that has learned "Close lives in the top right" and one pointing at the taskbar both score 0.0. Distance separates them.

This is still **dev**, so it is allowed to inform tuning. It is not the ablation -- that is a separate held-out run reported once.


In [ ]:
# GUARDED (2026-08-30). Everything in this cell runs AFTER the adapter has
# been written to disk. A Kaggle commit only persists /kaggle/working if the
# notebook completes -- so an exception here would throw away the training
# run that just finished. Nothing below is allowed to raise.
import traceback

try:
    import json
    import random

    from transformers import AutoProcessor

    from computeruse.eval.vlm_grounder import evaluate_arm, summarize, summarize_diagnostics
    from computeruse.training.dataset import resolve_path
    from computeruse.training.prepare_dataset import TrainingExample

    processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
    model = trainer.model
    model.eval()

    dev_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/dev.jsonl")
    dev_examples = [
        TrainingExample(**json.loads(line))
        for line in dev_split_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    # evaluate_arm needs the original label rows for bbox_real/real_size/scale,
    # which TrainingExample doesn't carry.
    labels_path = resolve_path(KAGGLE_DATASET_ROOT, "labels.jsonl")
    labels_by_id = {
        row["id"]: row
        for row in (
            json.loads(line)
            for line in labels_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        )
    }

    print(f"evaluating {len(dev_examples)} dev examples (batch-1 generation, a few minutes on a T4)...")
    results = evaluate_arm(
        model, processor, processor.tokenizer, dev_examples, labels_by_id, KAGGLE_DATASET_ROOT
    )

    accuracy = summarize(results)
    diagnostics = summarize_diagnostics(results)
    print("")
    print(f"dev click accuracy: {accuracy['overall']:.1%}")
    print(f"parse rate:         {diagnostics['parse_rate']:.1%}")
    print(f"median distance:    {diagnostics['median_center_distance']:.0f} / 1000 units")
    print(f"mean distance:      {diagnostics['mean_center_distance']:.0f} / 1000 units")
    print("")
    print("per app:")
    for app in sorted(k for k in accuracy if k != "overall"):
        print(f"  {app:20s} {accuracy[app]:.1%}")
except Exception:
    print("!! this cell failed, but the run is NOT lost -- checkpoints are on disk.")
    print("!! commit still succeeds; re-run this analysis separately.")
    traceback.print_exc()


### Qualitative sample

Now that there's a number, a handful of actual predictions, sampled **randomly with a pinned seed** rather than taking the first five. Random so it's representative; seeded so it's the same five across runs and therefore comparable.


In [ ]:
# GUARDED (2026-08-30). Everything in this cell runs AFTER the adapter has
# been written to disk. A Kaggle commit only persists /kaggle/working if the
# notebook completes -- so an exception here would throw away the training
# run that just finished. Nothing below is allowed to raise.
import traceback

try:
    sample = random.Random(42).sample(results, k=min(8, len(results)))
    for r in sample:
        left, top, right, bottom = r.ground_truth_bbox_norm
        center = (round((left + right) / 2), round((top + bottom) / 2))
        dist = "unparseable" if r.center_distance is None else f"{r.center_distance:.0f} away"
        print(
            f"{r.app:18s} pred={str(r.predicted_point):14s} truth_center={str(center):14s} "
            f"{'HIT ' if r.hit else 'miss'} ({dist})"
        )
        print(f"{'':18s} raw={r.predicted_text!r}")
except Exception:
    print("!! this cell failed, but the run is NOT lost -- checkpoints are on disk.")
    print("!! commit still succeeds; re-run this analysis separately.")
    traceback.print_exc()


## 7. Zero-shot arm (no adapter, no training)

**Independent of everything above except cells 1-9** (GPU check, clone/install, dataset mount) -- this loads the *base* model fresh, with no LoRA adapter, and does not need `trainer` to exist. If you only want this arm, you can skip straight from cell 9 to here; no training run required.

Cheap relative to the training cells: one model download (shared with the training path if this session already did one) plus generation over the same three splits `eval/uia_only.py`'s real run used (`dev`, `test_same_app`, `test_held_out_app`) -- ~20-30 min of inference on a T4, not hours of training. Commit this run too (`Save & Run All`) so the report doesn't need to be regenerated later.

In [ ]:
# GUARDED (2026-08-30). Everything in this cell runs AFTER the adapter has
# been written to disk. A Kaggle commit only persists /kaggle/working if the
# notebook completes -- so an exception here would throw away the training
# run that just finished. Nothing below is allowed to raise.
import traceback

try:
    import json

    from transformers import AutoModelForImageTextToText, AutoProcessor

    from computeruse.dataset.registry import load_registry
    from computeruse.eval.report import build_report
    from computeruse.eval.vlm_grounder import MODEL_ID, evaluate_arm, summarize_diagnostics, to_eval_records
    from computeruse.training.dataset import resolve_path
    from computeruse.training.prepare_dataset import SPLITS, TrainingExample

    # No get_peft_model call -- this is the point: the raw pretrained model,
    # nothing fine-tuned. torch_dtype matches build_trainer's choice for a
    # consistent memory footprint, not because it matters for a frozen model.
    zero_shot_model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype="float16").to("cuda")
    zero_shot_model.eval()
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    labels_path = resolve_path(KAGGLE_DATASET_ROOT, "labels.jsonl")
    labels_by_id = {
        row["id"]: row
        for row in (json.loads(line) for line in labels_path.read_text(encoding="utf-8").splitlines() if line.strip())
    }

    # Same three splits eval/uia_only.py's real run scored -- dev +
    # test_same_app + test_held_out_app -- so the two arms are directly
    # comparable in the combined report. Zero-shot has no hyperparameters to
    # tune, so looking at the test splits here doesn't taint anything the way
    # it would for a run being actively adjusted.
    eval_splits = [s for s in SPLITS if s != "train"]
    examples = []
    for split in eval_splits:
        split_path = resolve_path(KAGGLE_DATASET_ROOT, f"splits/{split}.jsonl")
        examples += [
            TrainingExample(**json.loads(line))
            for line in split_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

    print(f"evaluating {len(examples)} examples (dev + test_same_app + test_held_out_app) zero-shot...")
    zero_shot_results = evaluate_arm(
        zero_shot_model, processor, processor.tokenizer, examples, labels_by_id, KAGGLE_DATASET_ROOT
    )

    diagnostics = summarize_diagnostics(zero_shot_results)
    print(f"parse rate:      {diagnostics['parse_rate']:.1%}")
    print(f"median distance: {diagnostics['median_center_distance']:.0f} / 1000 units")

    # apps.yaml is tracked *source* (the app registry), not generated data. The
    # upload zip is built strictly from labels.jsonl's referenced paths, so it
    # contains no apps.yaml at all -- verified: 205 entries, images/ +
    # labels.jsonl + splits/ only. It arrives with the repo clone instead.
    # This line previously read it from KAGGLE_DATASET_ROOT and would have
    # raised right here, after the ~4GB model download had already run.
    _apps_yaml = REPO_DIR / "data" / "gui_grounding" / "apps.yaml"
    assert _apps_yaml.exists(), f"apps.yaml missing at {_apps_yaml} -- did the clone cell run?"
    richness_by_app = {c.name: c.richness for c in load_registry(_apps_yaml)}
    zero_shot_records = to_eval_records(zero_shot_results, arm="zero_shot", richness_by_app=richness_by_app)
    print(json.dumps(build_report(zero_shot_records), indent=2))

except Exception:
    print("!! this cell failed, but the run is NOT lost -- checkpoints are on disk.")
    print("!! commit still succeeds; re-run this analysis separately.")
    traceback.print_exc()


## Next steps (not this notebook)

1. ~~Collect the 6 blocked apps~~ -- done 2026-07-16, then re-verified/re-collected clean 2026-07-20 after the wrong-window and Notepad session-restore fixes: all 14/14 registry apps collected, v3 dataset frozen.
2. Build the 4 evaluation arms (UIA-only, zero-shot VLM, fine-tuned grounder, hybrid) -- UIA-only run for real 2026-08-16 (99.8% -- expected ceiling, see eval/uia_only.py's docstring on why this arm can't be low by construction); zero-shot cell 21 above is ready to run whenever there's a GPU session; hybrid combiner (eval/hybrid.py) is written and unit-tested, just needs both sides' real results; fine-tuned still needs this notebook's own checkpoint to actually finish and survive a commit.
3. Once all four have real results: join them with `eval/report.build_report` and report H1-H3, sliced 3 ways, honestly.


In [ ]:
# Final check: what is actually about to be persisted as this notebook
# version's Output. If you can see checkpoint dirs in the committed log,
# they exist -- and they are what the *next* chunk attaches as Input.
from pathlib import Path

working = Path('/kaggle/working')
total = 0
print('contents of /kaggle/working that will be saved as Output:')
print()
for d in sorted(working.rglob('checkpoint-*')):
    if d.is_dir():
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file())
        total += size
        print(f'  {d.relative_to(working)}  {size / 1e6:.0f} MB')

final = working / 'lora_grounder' / 'final'
if final.exists():
    size = sum(f.stat().st_size for f in final.rglob('*') if f.is_file())
    total += size
    print(f'  {final.relative_to(working)}  {size / 1e6:.0f} MB   <- trained adapter')

if total == 0:
    print('  NOTHING FOUND. Training did not reach step 100, or OUTPUT_DIR is wrong.')
else:
    print()
    print(f'total: {total / 1e6:.0f} MB (Kaggle output ceiling is 20 GB)')

print()
print('NEXT CHUNK:')
print('  1. right panel -> + Add Input -> Your Work -> this notebook')
print('  2. pick the version THIS run just created')
print('  3. detach the older version if one is still attached')
print('  4. Save & Run All (Commit) again')
print()
print("Cell 11 prints what it finds. If a later chunk says 'starting fresh")
print("from step 0', the attach did not take -- stop and redo it.")